In [37]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [38]:
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer

In [3]:
from summft.data import len_stats

len_stats()

--------------dialogue--------------
mean = 140.8 
median = 114.0
95th = 342.0
99th = 490.0
max = 1049
--------------summary--------------
mean = 25.8 
median = 23.0
95th = 54.0
99th = 67.0
max = 86
Number of samples: 14731


In [39]:
model_name = "Qwen/Qwen2.5-0.5B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(model_name)

In [40]:
dataset = load_dataset("knkarthick/samsum")
example = dataset["train"][0]

In [88]:
def format_example(example):
    messages = (
        {
            "role": "system",
            "content": "Summarize this dialogue in one or two sentences.",
        },
        {"role": "user", "content": example["dialogue"]},
        {"role": "assistant", "content": example["summary"]},
    )

    return messages


systems, prompts, answers = [], [], []
for i in range(7):
    system, prompt, answer = format_example(dataset["train"][i])
    systems.append(system)
    prompts.append(prompt)
    answers.append(answer)
for i in range(4):
    temp_list = [systems[i]]
    # for j in (4, 5, 6):
    #    temp_list.extend([prompts[j], answers[j]])
    temp_list.append(prompts[i])
    prompts[i] = temp_list
for i in range(4):
    print(f"---------Example {i + 1}---------")
    for turn in prompts[i]:
        print(turn["content"])
    print()
    print(answers[i]["content"])
    print()

---------Example 1---------
Summarize this dialogue in one or two sentences.
Amanda: I baked  cookies. Do you want some?
Jerry: Sure!
Amanda: I'll bring you tomorrow :-)

Amanda baked cookies and will bring Jerry some tomorrow.

---------Example 2---------
Summarize this dialogue in one or two sentences.
Olivia: Who are you voting for in this election? 
Oliver: Liberals as always.
Olivia: Me too!!
Oliver: Great

Olivia and Olivier are voting for liberals in this election. 

---------Example 3---------
Summarize this dialogue in one or two sentences.
Tim: Hi, what's up?
Kim: Bad mood tbh, I was going to do lots of stuff but ended up procrastinating
Tim: What did you plan on doing?
Kim: Oh you know, uni stuff and unfucking my room
Kim: Maybe tomorrow I'll move my ass and do everything
Kim: We were going to defrost a fridge so instead of shopping I'll eat some defrosted veggies
Tim: For doing stuff I recommend Pomodoro technique where u use breaks for doing chores
Tim: It really helps
Kim

In [14]:
model = AutoModelForCausalLM.from_pretrained(model_name)

Loading weights: 100%|██████████| 290/290 [00:00<00:00, 5960.94it/s]


In [ ]:
responses = []
for prompt in prompts[:4]:
    inputs = tokenizer.apply_chat_template(
        [prompt],
        add_generation_prompt=True,
        return_tensors="pt",
        return_dict=True,
    )
    out = model.generate(**inputs, max_new_tokens=100, do_sample=False)
    new_tokens = out[0][inputs["input_ids"].shape[1] :]
    response = tokenizer.decode(new_tokens, skip_special_tokens=True)
    responses.append(response)
print(f"Total of {len(responses)} responses.")

Total of 4 responses.


In [90]:
for response, answer in zip(responses, answers):
    print("Decoder summary:")
    print(response)
    print("\nHuman summary:")
    print(answer["content"])
    print("-" * 10)

Decoder summary:
Amanda and Jerry are planning to bake cookies together for a friend's birthday party on the following day.

Human summary:
Amanda baked cookies and will bring Jerry some tomorrow.
----------
Decoder summary:
Olivia and Oliver are both voting for the same political party, which is the Liberal Party.

Human summary:
Olivia and Olivier are voting for liberals in this election. 
----------
Decoder summary:
Tim asks Kim about her current mood and plans, while Kim mentions she was planning to do various tasks but ended up procrastinating. Tim suggests the Pomodoro Technique as an effective way to manage time more efficiently.

Human summary:
Kim may try the pomodoro technique recommended by Tim to get more stuff done.
----------
Decoder summary:
Rachel is outside and Edward thinks he's in over his head.

Human summary:
Edward thinks he is in love with Bella. Rachel wants Edward to open his door. Rachel is outside. 
----------
